# Colab setup: nr-jamming-localization

Run these cells top to bottom at the start of **every** Colab session. They clone (or update)
the repo, install it into Colab's Python, and point generated data/results at Google Drive.

**One-time setup**
1. **GitHub token.** github.com → Settings → Developer settings → Personal access tokens →
   *Fine-grained tokens* → *Generate new token*. Repository access: *Only select repositories* →
   `nr-jamming-localization`. Permissions → *Contents*: **Read and write** (read-only is enough if
   you never push from Colab).
2. **Colab secret.** In Colab's left sidebar open 🔑 *Secrets*, add `GITHUB_TOKEN` with the token
   as its value, and switch on *Notebook access*.
3. **GPU.** *Runtime → Change runtime type → T4 GPU* (or better).

In [ ]:
# 1. Check the GPU
!nvidia-smi -L || echo "No GPU attached: Runtime > Change runtime type > GPU"

In [ ]:
# 2. Clone the repo, or pull the latest version if it is already here
import os
import subprocess

from google.colab import userdata

REPO = "erhankarakoca/nr-jamming-localization"
ROOT = "/content/nr-jamming-localization"


def git(*args):
    r = subprocess.run(["git", *args], check=False, capture_output=True, text=True)
    print(r.stdout + r.stderr, end="")
    r.check_returncode()


# Git reads the token from the environment, so it never appears in a URL or in cell output.
os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
git("config", "--global", "credential.helper",
    '!f() { echo username=x-access-token; echo "password=$GITHUB_TOKEN"; }; f')
git("config", "--global", "user.name", "Erhan Karakoca")
git("config", "--global", "user.email", "64256175+erhankarakoca@users.noreply.github.com")

if os.path.isdir(f"{ROOT}/.git"):
    git("-C", ROOT, "pull", "--ff-only")
else:
    git("clone", f"https://github.com/{REPO}.git", ROOT)
os.chdir(ROOT)
git("log", "--oneline", "-3")

In [ ]:
# 3. Install the project into Colab's Python (keeps Colab's CUDA build of PyTorch)
!pip install -q -e . pytest

import sys

sys.path.insert(0, f"{ROOT}/src")  # an editable install is otherwise only seen after a restart

from importlib.metadata import version

import numpy
import torch

stale = [m.__name__ for m in (numpy, torch)
         if version(m.__name__).split("+")[0] != m.__version__.split("+")[0]]
if stale:
    print(f"pip upgraded {stale}: Runtime > Restart session, then run all cells again.")
else:
    import sionna.phy

    print(f"torch {torch.__version__} | sionna {version('sionna-no-rt')} "
          f"| device {sionna.phy.config.device}")

%load_ext autoreload
%autoreload 2

In [ ]:
# 4. Keep generated data and results on Google Drive (the Colab VM is wiped after the session)
from google.colab import drive

drive.mount("/content/drive")
STORE = "/content/drive/MyDrive/nr-jamming-localization"
os.environ["JAMLOC_DATA_DIR"] = f"{STORE}/data"
os.environ["JAMLOC_RESULTS_DIR"] = f"{STORE}/results"

from jamloc import paths

print("data    ->", paths.data_dir())
print("results ->", paths.results_dir())

In [ ]:
# 5. Smoke test
!python -m pytest -q

## Working from Colab

- **Get the latest code:** re-run cell 2 (it does `git pull`). With `autoreload` on, changed
  modules are picked up without restarting.
- **Recommended split:** write code on your computer and push; use Colab to run the heavy jobs
  (dataset generation, training). Anything saved via `jamloc.paths` lands on Drive.
- **If you do edit files here**, commit and push before the session ends:
  `!git add -A && git commit -m "..." && git push`
- Add new cells below, or make a copy of this notebook for an experiment.